# Phishing Email Classifier

Classify synthetic email text and identify the terms that most strongly influence phishing predictions.

**Safety and scope:** This project uses synthetic, non-sensitive telemetry for defensive analytics. It does not perform exploitation or execute malicious content.

## Goal

Build a transparent bag-of-words Naive Bayes classifier and inspect its errors and indicative terms.


## Setup

The notebook is deterministic, runs offline, and implements the core analytical method directly with NumPy and Pandas so the modeling logic remains inspectable.


In [1]:
import re
from collections import Counter

import numpy as np
import pandas as pd

SEED = 42
rng = np.random.default_rng(SEED)
pd.set_option("display.max_colwidth", 90)
pd.set_option("display.width", 120)

TOKEN_PATTERN = re.compile(r"[a-z0-9]{2,}")

def tokenize(text):
    return TOKEN_PATTERN.findall(text.lower())

def build_vocabulary(documents, min_count=2):
    counts = Counter(token for document in documents for token in tokenize(document))
    terms = sorted(term for term, count in counts.items() if count >= min_count)
    return {term: index for index, term in enumerate(terms)}

def count_matrix(documents, vocabulary):
    matrix = np.zeros((len(documents), len(vocabulary)), dtype=float)
    for row, document in enumerate(documents):
        for token in tokenize(document):
            if token in vocabulary:
                matrix[row, vocabulary[token]] += 1.0
    return matrix

def fit_multinomial_nb(features, labels, alpha=1.0):
    classes = np.array(sorted(np.unique(labels)))
    log_priors = []
    log_likelihoods = []
    for label in classes:
        class_rows = features[labels == label]
        token_totals = class_rows.sum(axis=0) + alpha
        log_likelihoods.append(np.log(token_totals / token_totals.sum()))
        log_priors.append(np.log(len(class_rows) / len(features)))
    return classes, np.asarray(log_priors), np.asarray(log_likelihoods)

def predict_multinomial_nb(features, model):
    classes, log_priors, log_likelihoods = model
    scores = features @ log_likelihoods.T + log_priors
    score_shift = scores - scores.max(axis=1, keepdims=True)
    probabilities = np.exp(score_shift)
    probabilities /= probabilities.sum(axis=1, keepdims=True)
    return classes[scores.argmax(axis=1)], probabilities

def classification_metrics(labels, predictions):
    labels = np.asarray(labels)
    predictions = np.asarray(predictions)
    tp = int(((labels == 1) & (predictions == 1)).sum())
    tn = int(((labels == 0) & (predictions == 0)).sum())
    fp = int(((labels == 0) & (predictions == 1)).sum())
    fn = int(((labels == 1) & (predictions == 0)).sum())
    precision = tp / max(tp + fp, 1)
    recall = tp / max(tp + fn, 1)
    f1 = 2 * precision * recall / max(precision + recall, 1e-12)
    return pd.Series({
        "accuracy": (tp + tn) / max(len(labels), 1),
        "precision": precision,
        "recall": recall,
        "f1": f1,
        "tp": tp,
        "fp": fp,
        "tn": tn,
        "fn": fn,
    })


## Steps

### 1. Build a synthetic email corpus


In [2]:
phishing_templates = [
    "urgent verify your account password now",
    "invoice overdue open attachment and confirm payment",
    "security alert login immediately using this link",
    "payroll update submit credentials before deadline",
    "shared document requires sign in to continue",
    "gift card request keep this confidential and act fast",
]
safe_templates = [
    "team meeting agenda and project notes attached",
    "monthly security newsletter with training schedule",
    "approved invoice summary available in finance portal",
    "engineering update deployment completed successfully",
    "benefits enrollment information from human resources",
    "customer report reviewed and ready for discussion",
]
noise_terms = ["quarterly", "review", "today", "internal", "update", "please", "notice", "document"]

messages = []
labels = []
for index in range(420):
    label = index % 2
    template_pool = phishing_templates if label else safe_templates
    template = rng.choice(template_pool)
    noise = " ".join(rng.choice(noise_terms, size=rng.integers(1, 4), replace=False))
    messages.append(f"{template} {noise}")
    labels.append(label)

email_data = pd.DataFrame({"text": messages, "phishing": labels})
shuffled = rng.permutation(len(email_data))
split_at = int(len(email_data) * 0.75)
train_rows, test_rows = shuffled[:split_at], shuffled[split_at:]

print("Corpus shape:", email_data.shape)
print("Phishing rate:", email_data["phishing"].mean())
print(email_data.sample(6, random_state=SEED).to_string(index=False))


Corpus shape: (420, 2)
Phishing rate: 0.5
                                                                       text  phishing
               gift card request keep this confidential and act fast notice         1
              benefits enrollment information from human resources internal         0
payroll update submit credentials before deadline notice document quarterly         1
                           urgent verify your account password now internal         1
       approved invoice summary available in finance portal review document         0
               gift card request keep this confidential and act fast review         1


### 2. Train and inspect the text model


In [3]:
train_text = email_data.loc[train_rows, "text"].tolist()
test_text = email_data.loc[test_rows, "text"].tolist()
train_labels = email_data.loc[train_rows, "phishing"].to_numpy(int)
test_labels = email_data.loc[test_rows, "phishing"].to_numpy(int)

vocabulary = build_vocabulary(train_text, min_count=2)
train_matrix = count_matrix(train_text, vocabulary)
test_matrix = count_matrix(test_text, vocabulary)
phishing_model = fit_multinomial_nb(train_matrix, train_labels)
predicted_label, predicted_probability = predict_multinomial_nb(test_matrix, phishing_model)
phishing_metrics = classification_metrics(test_labels, predicted_label)

terms = np.array(sorted(vocabulary, key=vocabulary.get))
log_odds = phishing_model[2][1] - phishing_model[2][0]
indicative_terms = pd.DataFrame({
    "term": terms,
    "phishing_log_odds": log_odds,
}).sort_values("phishing_log_odds", ascending=False)

scored_messages = email_data.loc[test_rows, ["text", "phishing"]].copy()
scored_messages["predicted"] = predicted_label
scored_messages["phishing_probability"] = predicted_probability[:, 1]

print("Test metrics:")
print(phishing_metrics.round(3).to_string())
print("\nMost phishing-indicative terms:")
print(indicative_terms.head(12).round(3).to_string(index=False))
print("\nSample scored messages:")
print(scored_messages.head(8).round(3).to_string(index=False))


Test metrics:
accuracy      1.0
precision     1.0
recall        1.0
f1            1.0
tp           50.0
fp            0.0
tn           55.0
fn            0.0

Most phishing-indicative terms:
    term  phishing_log_odds
    this              3.824
continue              3.357
      to              3.357
    sign              3.357
  shared              3.357
requires              3.357
 account              3.326
  verify              3.326
  urgent              3.326
password              3.326
     now              3.326
    your              3.326

Sample scored messages:
                                                                      text  phishing  predicted  phishing_probability
                     shared document requires sign in to continue document         1          1                   1.0
     team meeting agenda and project notes attached please update internal         0          0                   0.0
               approved invoice summary available in finance porta

## Checks


In [4]:
assert len(vocabulary) >= 25
assert phishing_metrics["f1"] >= 0.90
assert scored_messages["phishing_probability"].between(0, 1).all()
assert {"urgent", "credentials", "password"} & set(indicative_terms.head(15)["term"])
print("Checks passed: adequate vocabulary, strong synthetic-data F1, and interpretable phishing terms.")


Checks passed: adequate vocabulary, strong synthetic-data F1, and interpretable phishing terms.


## Next Steps

        - Add sender reputation, URL, attachment, and header-authentication features.
- Evaluate on time-separated data to avoid template leakage.
- Route uncertain predictions to analysts instead of auto-blocking them.
